# Intrinsics Check: Z4 vs CCD Height Map (v3)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-20  
**Last Modified:** 2026-04-21  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Z4, Defocus, CCD Height Map, Full Array Mode  

## Description

Check whether the Z4 intrinsic wavefront map (as produced by the AOS pipeline and
stored in the FAM mktable HDF5 output) correctly accounts for the focal plane
CCD-to-CCD height variation, and compare against multiple independent intrinsic
computations.

Key functionality:
1. Read the FAM donuts + visits HDF5 produced by `intrinsics_mktable`.
2. Read the matching `_fits.parquet` file with the per-visit linear (k1, k2, k3) fit coefficients.
3. Subtract the linear (tilt/tip + piston) portion of Z4 per donut using OCS field angles.
4. Bin the linear-corrected Z4 in focal plane (fpx, fpy) and plot the median map.
5. Build several per-donut Z4 intrinsics to compare:
   - **Custom** — `ts_wep`'s `Instrument.getIntrinsicZernikes` evaluated **per CCD** on a CCS field-angle grid inside each CCD's footprint.
   - **Batoid v2** — `batoid_rubin`'s `LSSTBuilder.build_det` evaluated per CCD via `batoid.zernike`, using the recipe from `Intrinsic Zk.ipynb`.
   - **Pipeline** — the per-donut `zk_intrinsic` already stored in the HDF5 (from the AOS pipeline).
   - **Transposed pipeline** — diagnostic: the pipeline intrinsic read at the position obtained by swapping x ↔ y around each CCD's bbox center, to test for a suspected fpx/fpy axis swap in the pipeline's intrinsic-Zernike interpolation.
6. For each intrinsic show a (fpx, fpy) median map of the intrinsic itself and of `(Z4 − linear) − intrinsic`, then present a block of intrinsic-vs-intrinsic comparisons (difference maps, hexbin scatters, residual histograms).
7. Both `ts_wep` and `batoid_rubin` already incorporate the per-CCD focal-plane height map inside their optics model (`batoid_rubin.LSSTBuilder` takes a `ccd_height_map_dir` parameter that defaults to `ccd_height_map`), so the **Custom** and **Batoid v2** intrinsics contain the height correction natively — we do **not** add an extra 15 μm/mm height term. `HEIGHT_TO_Z4_UM_PER_MM × height_mm` is kept only as a diagnostic quantity for the "pipeline − custom vs height" hexbin.

**Output:** In-notebook plots and a single PDF at `PDF_OUTPUT` with all figures.

**Based on:**
- `~/Astrophysics/Code/Claude/Intrinsic Zk.ipynb` — `batoid_rubin` per-CCD intrinsic recipe
- `~/Astrophysics/Code/Claude/intrinsic_wavefront.ipynb` — `ts_wep` `Instrument.getIntrinsicZernikes`
- `~/Astrophysics/Code/Claude/plot_Z_FAM_August-archive.ipynb` — height map reader, Z4↔height conversion (diagnostic)
- `intrinsics_mktable.ipynb` / `code/intrinsics_lib.py` — HDF5 donut table format
- `code/dz_fitting.py` — focal-plane Zernike basis used by the `_fits.parquet` fits

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-20 | Aaron Roodman | Initial version — radial i-band polynomial Z4 intrinsic + KNN height map |
| 2026-04-20 | Aaron Roodman | v2: replaced radial polynomial with `ts_wep` `Instrument.getIntrinsicZernikes` evaluated on a field-angle grid and interpolated; wrapped the custom intrinsic in `build_custom_z4_intrinsic`. Added HDF5 loader via `QTable.read`. Added per-CCD transposed-pipeline diagnostic. |
| 2026-04-21 | Aaron Roodman | v3: ts_wep intrinsic now built **per CCD** (one `Instrument` per detector, evaluated on a grid inside that CCD's CCS footprint) instead of reusing one fixed detector for the whole focal plane. Added `Batoid v2` intrinsic via `batoid_rubin.LSSTBuilder.build_det` following the recipe in `Intrinsic Zk.ipynb`. Results section reorganized: one block per intrinsic (map + residual vs linear-corrected data), followed by intrinsic-vs-intrinsic comparisons. Fixed the duplicate inline-plot bug (`_collect` returns `None`). Point `batoid_rubin` at a writable `BATOID_RUBIN_DATA_DIR` (default `~/batoid_rubin_data`) so the FEA/bend/CCD-height-map datasets can be downloaded from Zenodo on read-only conda installs. Corrected assumption that the optics models did not include the CCD height map — both ts_wep and batoid_rubin already apply it via `LSSTBuilder.ccd_height_map_dir`, so the extra 15 μm/mm additive term was removed from the Custom and Batoid v2 intrinsics (kept as a diagnostic variable only). |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Input HDF5 from intrinsics_mktable (streaming format)
INPUT_HDF5 = 'output/aos_fam_danish_triplets_20260315_20260317.hdf5'

# Corresponding linear-fit parquet (auto-derived as {stem}_fits.parquet if None)
FIT_PARQUET = None

# CCD focal plane height map (on USDF, copied under notebooks/rubin-work/aos/output/)
HEIGHT_MAP_FITS = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'

# Coordinate system used for the linear fit (must match the fit parquet)
COORD_SYS = 'OCS'

# Focal-plane radius used for normalizing the Zernike basis in the fit
# (matches default in code/dz_fitting.py: focal_plane_zernike_basis)
FP_RADIUS_DEG = 1.75

# Per-CCD intrinsic grid: evaluate the optics model (ts_wep Instrument or
# batoid directly) on a grid of field angles inside each CCD's CCS footprint
# and build a per-CCD LinearNDInterpolator. 8x8 (=64) points per CCD matches
# the recipe in 'Intrinsic Zk.ipynb'.
INTRINSIC_PER_CCD_GRID_N = 8

# Photometric band for the intrinsic (BandLabel.LSST_I by default; see below).
INTRINSIC_BAND_NAME = 'LSST_I'

# Batoid v2 intrinsic config — from 'Intrinsic Zk.ipynb'.
BATOID_V2_YAML = 'LSST_r.yaml'     # fiducial batoid model
BATOID_V2_WAVELENGTH_M = 620e-9    # 620 nm reference
BATOID_V2_JMAX = 28                # highest Noll index to compute
BATOID_V2_EPS = 0.612              # fractional obstruction
BATOID_V2_NX = 63                  # pupil sampling

# Writable data directory for batoid_rubin FEA / bending-mode / CCD-height-map
# datasets. The default path inside the installed conda env is read-only, so we
# set the BATOID_RUBIN_DATA_DIR env var to this path in the setup cell (only
# if it isn't already set in the shell). Override this parameter to point at
# a shared location you prefer.
BATOID_RUBIN_DATA_DIR = '~/batoid_rubin_data'

# Height-to-Z4 diagnostic: μm of Z4 per mm of local piston (height).
# Guillem's estimate (≈0.15 μm Z4 per 10 μm defocus = 15 μm/mm) — same
# conversion the old plot_Z_FAM notebook used. Kept here as a *diagnostic
# only* — both ts_wep's Instrument.getIntrinsicZernikes and batoid_rubin's
# per-CCD build_det already incorporate the focal-plane height map in their
# optics model (LSSTBuilder has a ccd_height_map_dir parameter), so the
# "Custom" and "Batoid v2" Z4 intrinsics do NOT add an extra height term.
HEIGHT_TO_Z4_UM_PER_MM = 15.0

# Binning for the focal-plane (fpx, fpy) median maps
FP_NSTEPS = 257           # number of bin edges (fine pitch ≈ 2.5 mm/bin)
FP_MAX_MM = 320.0         # half-range in mm

# Whether to restrict to donuts whose intra and extra centroids matched
REQUIRE_MATCHED = True

# Output PDF: all plots from the Results section go here
PDF_OUTPUT = 'output/intrinsics_checkZ4.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1

from astropy.io import fits
from astropy.table import QTable, Table
from scipy.stats import binned_statistic_2d
from scipy.interpolate import LinearNDInterpolator
from sklearn.neighbors import KNeighborsRegressor
import tqdm

# LSST camera (for pixel → focal-plane transform + per-CCD footprints)
from lsst.obs.lsst import LsstCam
from lsst.afw.cameraGeom import FIELD_ANGLE

# ts_wep — source of the intrinsic Zernike model
from lsst.ts.wep.task import (
    CalcZernikesTask, CalcZernikesTaskConfig, EstimateZernikesDanishTask,
)
from lsst.ts.wep.utils import getTaskInstrument, BandLabel

# Point batoid_rubin at a writable data dir before importing it — the
# default path inside the installed conda env is read-only, and
# LSSTBuilder downloads FEA / bending-mode / CCD-height-map datasets from
# Zenodo on first use. batoid_rubin's resolve_data_dir respects the
# BATOID_RUBIN_DATA_DIR env var; setdefault preserves any value the user
# already set in the shell.
os.environ.setdefault(
    'BATOID_RUBIN_DATA_DIR',
    os.path.expanduser(BATOID_RUBIN_DATA_DIR),
)
Path(os.environ['BATOID_RUBIN_DATA_DIR']).mkdir(parents=True, exist_ok=True)
print(f"BATOID_RUBIN_DATA_DIR = {os.environ['BATOID_RUBIN_DATA_DIR']}")

import batoid
from batoid_rubin import LSSTBuilder

# Shared cameraGeom helpers (common/camera_utils.py)
sys.path.insert(0, str(Path.cwd().parent))
from common.camera_utils import pixel_to_focal  # noqa: E402

plt.rcParams['figure.dpi'] = 110

<a id='functions'></a>
## Helper Functions

In [ ]:
def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Attach a vertical colorbar that matches the host axes' height."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes('right', size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ------------------------------------------------------------------
# Donut → focal-plane coordinates
# ------------------------------------------------------------------

def compute_fp_coords(aosTable, camera, x_col='centroid_x_intra',
                      y_col='centroid_y_intra'):
    """Evaluate fpx/fpy in mm for every donut using per-detector transforms.

    Groups donuts by detector and calls ``common.camera_utils.pixel_to_focal``
    once per sensor (vectorized over all donuts on that sensor).

    Returns
    -------
    fpx_mm, fpy_mm : ndarray
    """
    det_names = np.asarray(aosTable['detector']).astype(str)
    x_pix = np.asarray(aosTable[x_col], dtype=float)
    y_pix = np.asarray(aosTable[y_col], dtype=float)
    fpx = np.full_like(x_pix, np.nan)
    fpy = np.full_like(y_pix, np.nan)
    for det in camera:
        name = det.getName()
        mask = (det_names == name)
        if not np.any(mask):
            continue
        fx, fy = pixel_to_focal(x_pix[mask], y_pix[mask], det)
        fpx[mask] = fx
        fpy[mask] = fy
    return fpx, fpy


def ccd_center_fp(camera):
    """Return {detector_name: (cx_mm, cy_mm)} for every detector in `camera`."""
    centers = {}
    for det in camera:
        c = det.getBBox().getCenter()  # lsst.geom.Point2D
        cx, cy = pixel_to_focal(np.array([c.getX()]),
                                np.array([c.getY()]), det)
        centers[det.getName()] = (float(cx[0]), float(cy[0]))
    return centers


def _per_ccd_field_angle_grid(det, grid_n):
    """Build an n×n CCS field-angle grid (radians) covering one CCD's footprint.

    Mirrors the DVCS→CCS transpose used in 'Intrinsic Zk.ipynb':
    CCS x = DVCS y of the corners, CCS y = DVCS x.
    """
    corners = det.getCorners(FIELD_ANGLE)         # radians, DVCS
    xmin = min(c.y for c in corners)
    xmax = max(c.y for c in corners)
    ymin = min(c.x for c in corners)
    ymax = max(c.x for c in corners)
    xs = np.linspace(xmin, xmax, grid_n + 1)
    ys = np.linspace(ymin, ymax, grid_n + 1)
    xs = 0.5 * (xs[1:] + xs[:-1])                 # half-cell inset from edges
    ys = 0.5 * (ys[1:] + ys[:-1])
    xx, yy = np.meshgrid(xs, ys)
    return xx.ravel(), yy.ravel()                 # radians (CCS)


# ------------------------------------------------------------------
# ts_wep per-CCD intrinsic
# ------------------------------------------------------------------

def _ts_wep_task_config():
    """Default CalcZernikesTask config matching intrinsics_lib.get_intrinsic_map."""
    config = CalcZernikesTaskConfig()
    config.estimateZernikes.retarget(EstimateZernikesDanishTask)
    config.donutStampSelector.maxSelect = 20
    config.donutStampSelector.maxFracBadPixels = 2.0e-4
    config.donutStampSelector.useCustomSnLimit = True
    config.donutStampSelector.minSignalToNoise = 100
    config.estimateZernikes.binning = 2
    config.estimateZernikes.saveHistory = False
    return config


def build_z4_intrinsic_per_ccd_ts_wep(camera, iZs, band=BandLabel.LSST_I,
                                      grid_n=INTRINSIC_PER_CCD_GRID_N,
                                      verbose=True):
    """Build one ts_wep Instrument per CCD, evaluate Z4 intrinsic on a per-CCD
    CCS field-angle grid, and return a dict of per-CCD interpolators.

    Each interpolator takes CCS field angles in **degrees** (to match the
    storage convention of thx_CCS/thy_CCS after ``np.rad2deg``) and returns
    Z4 in μm. The per-donut evaluator ``evaluate_z4_per_ccd`` dispatches by
    detector name.

    Parameters
    ----------
    camera : lsst.afw.cameraGeom.Camera
    iZs : sequence of int
        Noll indices matching the data (e.g. [4, 5, ..., 26]).
    band : BandLabel
    grid_n : int
        Number of grid points per axis inside each CCD footprint.
    verbose : bool

    Returns
    -------
    interp_dict : dict {det_name: fn(thx_deg, thy_deg) -> Z4 μm}
    """
    if 4 not in iZs:
        raise ValueError('iZs must contain Z4 (Noll index 4)')
    z4_col = list(iZs).index(4)
    noll_arr = np.asarray(iZs, dtype=int)

    task = CalcZernikesTask(config=_ts_wep_task_config())
    inst_config_path = task.estimateZernikes.config.instConfigFile

    interp_dict = {}
    it = camera
    if verbose:
        print(f'ts_wep per-CCD intrinsic: {grid_n}×{grid_n} grid per detector')
        it = tqdm.tqdm(list(camera), desc='ts_wep per CCD')
    for det in it:
        name = det.getName()
        instrument = getTaskInstrument('LSSTCam', name, inst_config_path)
        thx_rad, thy_rad = _per_ccd_field_angle_grid(det, grid_n)
        thx_deg = np.rad2deg(thx_rad)
        thy_deg = np.rad2deg(thy_rad)
        z4_um = np.empty(thx_deg.size)
        for i, (x, y) in enumerate(zip(thx_deg, thy_deg)):
            zk = instrument.getIntrinsicZernikes(
                xAngle=float(x), yAngle=float(y),
                defocalType=None, band=band, nollIndices=noll_arr,
            )
            z4_um[i] = zk[z4_col] * 1.0e6          # meters → μm
        interp = LinearNDInterpolator(
            np.column_stack([thx_deg, thy_deg]), z4_um, fill_value=np.nan
        )
        interp_dict[name] = (lambda _i: lambda tx, ty: _i(tx, ty))(interp)
    return interp_dict


# ------------------------------------------------------------------
# Batoid v2 per-CCD intrinsic (Intrinsic Zk.ipynb recipe)
# ------------------------------------------------------------------

def build_z4_intrinsic_per_ccd_batoid_v2(camera,
                                         yaml_name=BATOID_V2_YAML,
                                         wavelength_m=BATOID_V2_WAVELENGTH_M,
                                         grid_n=INTRINSIC_PER_CCD_GRID_N,
                                         jmax=BATOID_V2_JMAX,
                                         eps=BATOID_V2_EPS,
                                         nx=BATOID_V2_NX,
                                         verbose=True):
    """Build per-CCD Z4 intrinsic interpolators using batoid_rubin LSSTBuilder.

    Mirrors the recipe in ``Intrinsic Zk.ipynb``:

    - ``fiducial = batoid.Optic.fromYaml(yaml_name)``
    - ``LSSTBuilder(fiducial, dof_coord_system='OCS',
                    flip_m2_bending_modes=False, dof_angle_units='degree')``
    - For each detector: ``telescope = builder.build_det(det.getId())`` and
      ``batoid.zernike(telescope, theta_x, theta_y, wavelength,
                         projection='gnomonic', jmax, eps, nx)``

    The returned Zernike coefficients are in waves; multiplied by
    ``wavelength_m * 1e6`` they become μm.

    Returns
    -------
    interp_dict : dict {det_name: fn(thx_deg, thy_deg) -> Z4 μm}
    """
    fiducial = batoid.Optic.fromYaml(yaml_name)
    builder = LSSTBuilder(
        fiducial,
        dof_coord_system='OCS',
        flip_m2_bending_modes=False,
        dof_angle_units='degree',
    )
    scale_um = wavelength_m * 1.0e6

    interp_dict = {}
    it = camera
    if verbose:
        print(f'batoid_rubin v2 intrinsic: {grid_n}×{grid_n} grid per '
              f'detector (λ={wavelength_m*1e9:.1f} nm, yaml={yaml_name})')
        it = tqdm.tqdm(list(camera), desc='batoid v2 per CCD')
    for det in it:
        telescope = builder.build_det(det.getId())
        thx_rad, thy_rad = _per_ccd_field_angle_grid(det, grid_n)
        z4_um = np.empty(thx_rad.size)
        for i, (x, y) in enumerate(zip(thx_rad, thy_rad)):
            zk = batoid.zernike(
                telescope, theta_x=float(x), theta_y=float(y),
                wavelength=wavelength_m, projection='gnomonic',
                jmax=jmax, eps=eps, nx=nx,
            )
            z4_um[i] = zk[4] * scale_um            # waves * λ(m) * 1e6 = μm
        # Interpolate in degrees to match the ts_wep interface.
        thx_deg = np.rad2deg(thx_rad)
        thy_deg = np.rad2deg(thy_rad)
        interp = LinearNDInterpolator(
            np.column_stack([thx_deg, thy_deg]), z4_um, fill_value=np.nan
        )
        interp_dict[det.getName()] = (lambda _i: lambda tx, ty: _i(tx, ty))(interp)
    return interp_dict


def evaluate_z4_per_ccd(det_names, thx_deg, thy_deg, interp_dict):
    """Dispatch each donut to its CCD's per-CCD Z4 interpolator."""
    out = np.full_like(thx_deg, np.nan, dtype=float)
    for name, fn in interp_dict.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        out[mask] = fn(thx_deg[mask], thy_deg[mask])
    return out


# ------------------------------------------------------------------
# Transposed pipeline intrinsic (x↔y swap per CCD, diagnostic)
# ------------------------------------------------------------------

def build_transposed_pipeline_z4(fpx_mm, fpy_mm, det_names,
                                 Z4_pipeline_intrinsic_um,
                                 camera, keep_mask=None):
    """Transpose x↔y around each CCD's center, then look up the pipeline Z4
    intrinsic at the swapped position via nearest-neighbor over real donuts.
    See request-level discussion for details; returns per-donut transposed Z4
    plus the swapped positions and the CCD centers dict.
    """
    from scipy.spatial import cKDTree
    det_centers = ccd_center_fp(camera)
    fpx_swap = np.full_like(fpx_mm, np.nan)
    fpy_swap = np.full_like(fpy_mm, np.nan)
    for name, (cx, cy) in det_centers.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        dx = fpx_mm[mask] - cx
        dy = fpy_mm[mask] - cy
        fpx_swap[mask] = cx + dy
        fpy_swap[mask] = cy + dx
    if keep_mask is None:
        keep_mask = np.ones(len(fpx_mm), dtype=bool)
    ref_pts = np.column_stack([fpx_mm[keep_mask], fpy_mm[keep_mask]])
    ref_vals = Z4_pipeline_intrinsic_um[keep_mask]
    tree = cKDTree(ref_pts)
    _, nn_idx = tree.query(np.column_stack([fpx_swap, fpy_swap]))
    return ref_vals[nn_idx], fpx_swap, fpy_swap, det_centers


# ------------------------------------------------------------------
# Focal-plane height map
# ------------------------------------------------------------------

def make_metrology_table(file, rsid=None):
    """Build a per-point focal-plane metrology table from the height map FITS file.

    Mirrors the reader in plot_Z_FAM_August-archive.ipynb: each per-sensor BinTableHDU
    is concatenated into one table with columns (fpx, fpy, z_mod, z_meas, det).
    Note the x/y swap from CCS to focal-plane convention: fpx = Y_CCS, fpy = X_CCS.
    """
    rows = []
    with fits.open(file) as hdulist:
        for hdu in tqdm.tqdm(hdulist, desc='metrology HDUs'):
            if not isinstance(hdu, fits.BinTableHDU):
                continue
            extname = hdu.header.get('EXTNAME', '')
            if rsid is not None and extname != rsid:
                continue
            if rsid is None and not re.fullmatch(r'R\d\dS\d\d', extname):
                continue
            det_label = re.sub(r'(R\d\d)(S\d\d)', r'\1_\2', extname)
            tab = Table(hdu.data)
            for x, y, z_mod, z_meas in zip(
                tab['X_CCS'], tab['Y_CCS'], tab['Z_CCS_MODEL'], tab['Z_CCS_MEASURED']
            ):
                fpx = y
                fpy = x
                rows.append([fpx, fpy, z_mod, z_meas, det_label])
    return Table(rows=rows, names=['fpx', 'fpy', 'z_mod', 'z_meas', 'det'])


def get_height_interpolator(metrology_table, k=3, weight_type='distance', ztype='measured'):
    """KNN interpolator for focal-plane height given a metrology table."""
    x = np.column_stack((metrology_table['fpx'], metrology_table['fpy']))
    if ztype == 'measured':
        y = np.array(metrology_table['z_meas'])
    elif ztype == 'model':
        y = np.array(metrology_table['z_mod'])
    else:
        raise ValueError("ztype must be 'measured' or 'model'")
    knn = KNeighborsRegressor(n_neighbors=k, weights=weight_type)
    knn.fit(x, y)

    def interp_func(fpx, fpy):
        fpx = np.atleast_1d(fpx)
        fpy = np.atleast_1d(fpy)
        pts = np.column_stack((fpx, fpy))
        return knn.predict(pts)

    return interp_func


# ------------------------------------------------------------------
# Linear (k1, k2, k3) model evaluation
# ------------------------------------------------------------------

def linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG):
    """Evaluate the Z1-Z3 focal-plane Zernike linear model at each donut.

    Matches the basis in code/dz_fitting.focal_plane_zernike_basis:
        Z1 = 1            (piston)
        Z2 = 2 * x        (tilt), x = thx_deg / fp_radius
        Z3 = 2 * y        (tip),  y = thy_deg / fp_radius
    So the linear model value is:
        k1 * 1 + k2 * 2*x + k3 * 2*y  (μm)
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    return k1 + 2.0 * k2 * x + 2.0 * k3 * y


# ------------------------------------------------------------------
# Binning + plotting helpers
# ------------------------------------------------------------------

def bin_median_2d(xval, yval, zval, xbins, ybins):
    """Median of zval on an (xbins × ybins) 2D grid over (xval, yval)."""
    stat, x_edges, y_edges, _ = binned_statistic_2d(
        xval, yval, zval, statistic='median', bins=[xbins, ybins]
    )
    return stat, x_edges, y_edges


def plot_fp_map(stat, x_edges, y_edges, title, cmap='viridis',
                vmin=None, vmax=None, label=None):
    """Imshow a (fpx, fpy) 2D map with a matched colorbar."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        stat.T, origin='lower',
        extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
        cmap=cmap, interpolation='none', aspect='equal',
        vmin=vmin, vmax=vmax,
    )
    ax.set_xlabel('fpx [mm]')
    ax.set_ylabel('fpy [mm]')
    ax.set_title(title)
    cb = add_colorbar(im)
    if label is not None:
        cb.set_label(label)
    fig.tight_layout()
    return fig, ax

<a id='data'></a>
## Data Access

In [ ]:
# Resolve paths and load visits + donuts from HDF5 (astropy HDF5 format)
hdf5_path = Path(INPUT_HDF5)
if not hdf5_path.exists():
    raise FileNotFoundError(f'HDF5 not found: {hdf5_path}')

if FIT_PARQUET is None:
    fits_path = hdf5_path.parent / f'{hdf5_path.stem}_fits.parquet'
else:
    fits_path = Path(FIT_PARQUET)
if not fits_path.exists():
    raise FileNotFoundError(f'Fit parquet not found: {fits_path}')

height_path = Path(HEIGHT_MAP_FITS)
if not height_path.exists():
    raise FileNotFoundError(f'Height map not found: {height_path}')

print(f'HDF5:        {hdf5_path}')
print(f'Fit parquet: {fits_path}')
print(f'Height map:  {height_path}')

visit_info = QTable.read(str(hdf5_path), path='visits')
aosTable = QTable.read(str(hdf5_path), path='donuts')
print(f'Loaded {len(aosTable):,} donuts, {len(aosTable.colnames)} columns, '
      f'{len(visit_info)} visits')

In [ ]:
# Load the per-visit linear-fit coefficients (k1, k2, k3 for Z4)
fits_table = pd.read_parquet(fits_path)
print(f'Fit rows: {len(fits_table)}, columns: {len(fits_table.columns)}')
print([c for c in fits_table.columns if c.startswith('z1toz3_z4_')])

required = ['day_obs', 'seq_num',
            'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']
missing = [c for c in required if c not in fits_table.columns]
if missing:
    raise KeyError(f'Missing required fit columns: {missing}')

if 'z1toz3_bad_fit' in fits_table.columns:
    n_bad = int(fits_table['z1toz3_bad_fit'].sum())
    print(f'  bad_fit visits: {n_bad}/{len(fits_table)}')

<a id='analysis'></a>
## Analysis

In [ ]:
# Build donut-level arrays we will re-use
zk_col = f'zk_{COORD_SYS}'
zk_intr_col = f'zk_intrinsic_{COORD_SYS}'  # pipeline's per-donut intrinsic (μm)
thx_col = f'thx_{COORD_SYS}'
thy_col = f'thy_{COORD_SYS}'

zk_arr = np.stack(aosTable[zk_col])            # (n_donuts, n_Z)
nZk = zk_arr.shape[1]
iZs = list(range(4, 4 + nZk)) if nZk != 19 else list(range(4, 23))
if 'nollIndices' in visit_info.colnames:
    iZs_vi = [int(n) for n in visit_info['nollIndices'][0]]
    if len(iZs_vi) == nZk:
        iZs = iZs_vi
iZidx = {iZ: i for i, iZ in enumerate(iZs)}
print(f'Noll indices ({nZk} terms): {iZs}')
z4_col_idx = iZidx[4]

Z4_data_um = zk_arr[:, z4_col_idx]
Z4_pipeline_intrinsic_um = np.stack(aosTable[zk_intr_col])[:, z4_col_idx]

# Field angles (linear fits use these) — stored in radians
thx_deg = np.rad2deg(np.asarray(aosTable[thx_col], dtype=float))
thy_deg = np.rad2deg(np.asarray(aosTable[thy_col], dtype=float))

# fpx / fpy in mm — derived from per-detector pixel centroids via cameraGeom.
camera = LsstCam.getCamera()
fpx, fpy = compute_fp_coords(aosTable, camera,
                             x_col='centroid_x_intra',
                             y_col='centroid_y_intra')
print(f'fpx range [mm]: [{np.nanmin(fpx):+.1f}, {np.nanmax(fpx):+.1f}]; '
      f'fpy range [mm]: [{np.nanmin(fpy):+.1f}, {np.nanmax(fpy):+.1f}]')

# Optional selection
keep = np.ones(len(aosTable), dtype=bool)
if REQUIRE_MATCHED and 'matched_intra_extra' in aosTable.colnames:
    keep &= np.asarray(aosTable['matched_intra_extra'], dtype=bool)
print(f'Selecting {keep.sum():,}/{len(keep):,} donuts')

In [ ]:
# Merge per-visit (k1, k2, k3) coefficients onto the donut table and
# compute the linear-fit Z4 prediction per donut.

fit_keys = fits_table[['day_obs', 'seq_num',
                       'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']].copy()

donuts_df = pd.DataFrame({
    'day_obs': np.asarray(aosTable['day_obs'], dtype=int),
    'seq_num': np.asarray(aosTable['seq_num'], dtype=int),
})
merged = donuts_df.merge(fit_keys, on=['day_obs', 'seq_num'], how='left')

k1 = merged['z1toz3_z4_c1'].to_numpy()
k2 = merged['z1toz3_z4_c2'].to_numpy()
k3 = merged['z1toz3_z4_c3'].to_numpy()
missing_fit = np.isnan(k1) | np.isnan(k2) | np.isnan(k3)
print(f'Donuts missing a linear fit: {int(missing_fit.sum())}')
keep &= ~missing_fit

Z4_linear_um = linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG)
# The linear fit was performed on (Z4_data − Z4_pipeline_intrinsic); subtracting
# only Z4_linear from Z4_data keeps the field-dependent intrinsic shape in
# the map while removing per-image piston/tilt drift.
Z4_linear_corr_um = Z4_data_um - Z4_linear_um
print('Z4 statistics (μm):')
for label, arr in [('Z4_data', Z4_data_um),
                   ('Z4_linear_model', Z4_linear_um),
                   ('Z4 - linear', Z4_linear_corr_um),
                   ('Z4_pipeline_intrinsic', Z4_pipeline_intrinsic_um)]:
    a = arr[keep]
    print(f'  {label:24s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'n={len(a):,}')

In [ ]:
# Load the focal plane height map and build the KNN interpolator.
met_table = make_metrology_table(str(height_path))
print(f'Metrology points: {len(met_table):,}')
height_interp = get_height_interpolator(met_table, k=3, weight_type='distance',
                                        ztype='measured')

# Evaluate height at each donut (use intra-focal focal-plane position)
height_mm = height_interp(fpx, fpy)
print(f'Height range [mm]: [{np.nanmin(height_mm):+.4f}, {np.nanmax(height_mm):+.4f}]')

In [ ]:
# Build two per-CCD optics intrinsics and use them as the Z4 intrinsics.
#
#  Z4_custom_intrinsic    = ts_wep per-CCD Instrument.getIntrinsicZernikes
#  Z4_batoidv2_intrinsic  = batoid_rubin per-CCD build_det + batoid.zernike
#
# Both optics models already incorporate the focal-plane CCD height map
# (batoid_rubin's LSSTBuilder has a ccd_height_map_dir parameter that
# defaults to 'ccd_height_map', and ts_wep's Instrument uses the same
# batoid infrastructure internally), so we do NOT add a separate
# HEIGHT_TO_Z4_UM_PER_MM × height term. Z4_height_um is kept only as a
# diagnostic quantity for the hexbin plot.
band = getattr(BandLabel, INTRINSIC_BAND_NAME)
det_names_all = np.asarray(aosTable['detector']).astype(str)

# CCS field angles (degrees) for the per-CCD lookup. Each CCD has a fixed
# footprint in CCS (camera-fixed), so grouping donuts by detector and using
# the CCD's own optics model is well-defined.
thx_CCS_deg = np.rad2deg(np.asarray(aosTable['thx_CCS'], dtype=float))
thy_CCS_deg = np.rad2deg(np.asarray(aosTable['thy_CCS'], dtype=float))

interp_ts_wep = build_z4_intrinsic_per_ccd_ts_wep(
    camera, iZs, band=band, grid_n=INTRINSIC_PER_CCD_GRID_N,
)
Z4_custom_intrinsic_um = evaluate_z4_per_ccd(
    det_names_all, thx_CCS_deg, thy_CCS_deg, interp_ts_wep,
)

interp_batoid_v2 = build_z4_intrinsic_per_ccd_batoid_v2(
    camera,
    yaml_name=BATOID_V2_YAML,
    wavelength_m=BATOID_V2_WAVELENGTH_M,
    grid_n=INTRINSIC_PER_CCD_GRID_N,
)
Z4_batoidv2_intrinsic_um = evaluate_z4_per_ccd(
    det_names_all, thx_CCS_deg, thy_CCS_deg, interp_batoid_v2,
)

# Diagnostic only — not added to the intrinsics above.
Z4_height_um = HEIGHT_TO_Z4_UM_PER_MM * height_mm

print('Per-donut Z4 intrinsics (μm, keep-selected):')
for label, arr in [
    ('Custom (ts_wep per-CCD)',    Z4_custom_intrinsic_um),
    ('Batoid v2 (batoid per-CCD)', Z4_batoidv2_intrinsic_um),
    ('height term (diagnostic)',   Z4_height_um),
]:
    a = arr[keep]
    print(f'  {label:28s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'min={np.nanmin(a):+.3f}  max={np.nanmax(a):+.3f}')

In [ ]:
# Transposed pipeline Z4 intrinsic: for each donut, swap x↔y relative to
# its CCD center and read the pipeline intrinsic at the swapped position
# via nearest-neighbor lookup. Used to diagnose a suspected fpx↔fpy swap
# inside the pipeline's intrinsic-Zernike interpolation.
det_names_all = np.asarray(aosTable['detector']).astype(str)
Z4_pipeline_transposed_um, fpx_swap, fpy_swap, det_centers = \
    build_transposed_pipeline_z4(
        fpx, fpy, det_names_all, Z4_pipeline_intrinsic_um, camera,
        keep_mask=keep,
    )
print(f'{len(det_centers)} detector centers computed')
a = Z4_pipeline_transposed_um[keep]
print('Transposed pipeline Z4 intrinsic (μm):')
print(f'  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
      f'min={np.nanmin(a):+.3f}  max={np.nanmax(a):+.3f}')

<a id='results'></a>
## Results & Plots

In [ ]:
# Common binning used across all focal-plane maps + PDF figure collector.
xbins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)
ybins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)

xv = fpx[keep]
yv = fpy[keep]

# Every plot cell below calls _collect(fig); the final cell writes them all
# to PDF_OUTPUT. Re-running this cell starts a fresh collection (no stale
# pages). _collect returns None on purpose: returning the figure made the
# notebook render each plot twice inline (once via the matplotlib inline
# auto-show and once as the cell's last-expression repr).
_pdf_figs = []


def _collect(fig):
    """Remember a figure so it gets written to PDF_OUTPUT at the end."""
    _pdf_figs.append(fig)

### Data (linearly corrected Z4)

In [ ]:
# Baseline: median Z4 (linearly corrected by k1,k2,k3) in focal plane.
# Every intrinsic below is compared against this map.
stat_data, xe, ye = bin_median_2d(xv, yv, Z4_linear_corr_um[keep], xbins, ybins)
vabs_data = np.nanpercentile(np.abs(stat_data), 98)
fig, _ = plot_fp_map(
    stat_data, xe, ye,
    title='data: median Z4 − (k1, k2, k3) linear fit',
    cmap='RdBu_r', vmin=-vabs_data, vmax=vabs_data,
    label='Z4 [μm]',
)
_collect(fig)

### Intrinsic: Custom (ts_wep per-CCD; CCD height map built in)

In [ ]:
# Custom intrinsic (ts_wep per-CCD — CCD height map built in) + diff vs data.
stat_custom, _, _ = bin_median_2d(xv, yv, Z4_custom_intrinsic_um[keep], xbins, ybins)
vabs_c = np.nanpercentile(np.abs(stat_custom), 98)
fig, _ = plot_fp_map(
    stat_custom, xe, ye,
    title='custom Z4 intrinsic: ts_wep per-CCD (CCD height included)',
    cmap='RdBu_r', vmin=-vabs_c, vmax=vabs_c, label='Z4 [μm]',
)
_collect(fig)

diff_custom_per_donut = Z4_linear_corr_um - Z4_custom_intrinsic_um
stat_diff_custom, _, _ = bin_median_2d(xv, yv, diff_custom_per_donut[keep],
                                       xbins, ybins)
vabs_dc = np.nanpercentile(np.abs(stat_diff_custom), 98)
fig, _ = plot_fp_map(
    stat_diff_custom, xe, ye,
    title='(Z4 − linear)  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_dc, vmax=vabs_dc, label='ΔZ4 [μm]',
)
_collect(fig)

### Intrinsic: Batoid v2 (batoid_rubin per-CCD; CCD height map built in)

In [ ]:
# Batoid v2 intrinsic (batoid_rubin per-CCD — CCD height map built in) + diff vs data.
stat_bv2, _, _ = bin_median_2d(xv, yv, Z4_batoidv2_intrinsic_um[keep], xbins, ybins)
vabs_bv2 = np.nanpercentile(np.abs(stat_bv2), 98)
fig, _ = plot_fp_map(
    stat_bv2, xe, ye,
    title='Batoid v2 Z4 intrinsic: batoid_rubin per-CCD (CCD height included)',
    cmap='RdBu_r', vmin=-vabs_bv2, vmax=vabs_bv2, label='Z4 [μm]',
)
_collect(fig)

diff_bv2_per_donut = Z4_linear_corr_um - Z4_batoidv2_intrinsic_um
stat_diff_bv2, _, _ = bin_median_2d(xv, yv, diff_bv2_per_donut[keep],
                                    xbins, ybins)
vabs_dbv2 = np.nanpercentile(np.abs(stat_diff_bv2), 98)
fig, _ = plot_fp_map(
    stat_diff_bv2, xe, ye,
    title='(Z4 − linear)  −  Batoid v2 Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_dbv2, vmax=vabs_dbv2, label='ΔZ4 [μm]',
)
_collect(fig)

### Intrinsic: Pipeline (from HDF5 `zk_intrinsic`)

In [ ]:
# Pipeline intrinsic (from HDF5 zk_intrinsic) + its diff vs data.
stat_pipe, _, _ = bin_median_2d(xv, yv, Z4_pipeline_intrinsic_um[keep],
                                xbins, ybins)
vabs_p = np.nanpercentile(np.abs(stat_pipe), 98)
fig, _ = plot_fp_map(
    stat_pipe, xe, ye,
    title='pipeline Z4 intrinsic (from HDF5 zk_intrinsic)',
    cmap='RdBu_r', vmin=-vabs_p, vmax=vabs_p, label='Z4 [μm]',
)
_collect(fig)

diff_pipe_per_donut = Z4_linear_corr_um - Z4_pipeline_intrinsic_um
stat_diff_pipe, _, _ = bin_median_2d(xv, yv, diff_pipe_per_donut[keep],
                                     xbins, ybins)
vabs_dp = np.nanpercentile(np.abs(stat_diff_pipe), 98)
fig, _ = plot_fp_map(
    stat_diff_pipe, xe, ye,
    title='(Z4 − linear)  −  pipeline Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_dp, vmax=vabs_dp, label='ΔZ4 [μm]',
)
_collect(fig)

### Intrinsic: Transposed pipeline (x ↔ y per CCD, diagnostic)

In [ ]:
# Transposed pipeline intrinsic (x↔y swap per CCD) + its diff vs data.
stat_pipeT, _, _ = bin_median_2d(xv, yv, Z4_pipeline_transposed_um[keep],
                                 xbins, ybins)
vabs_pT = np.nanpercentile(np.abs(stat_pipeT), 98)
fig, _ = plot_fp_map(
    stat_pipeT, xe, ye,
    title='transposed pipeline Z4 intrinsic (swap x↔y per CCD)',
    cmap='RdBu_r', vmin=-vabs_pT, vmax=vabs_pT, label='Z4 [μm]',
)
_collect(fig)

diff_pipeT_per_donut = Z4_linear_corr_um - Z4_pipeline_transposed_um
stat_diff_pipeT, _, _ = bin_median_2d(xv, yv, diff_pipeT_per_donut[keep],
                                      xbins, ybins)
vabs_dpT = np.nanpercentile(np.abs(stat_diff_pipeT), 98)
fig, _ = plot_fp_map(
    stat_diff_pipeT, xe, ye,
    title='(Z4 − linear)  −  transposed pipeline Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_dpT, vmax=vabs_dpT, label='ΔZ4 [μm]',
)
_collect(fig)

### Intrinsic-vs-intrinsic comparisons

In [ ]:
# Intrinsic-vs-intrinsic difference maps (focal plane). All four intrinsics
# here already include the CCD height map (ts_wep and batoid_rubin build it
# in via batoid_rubin's ccd_height_map_dir; the pipeline's zk_intrinsic is
# what we are checking).

# ts_wep per-CCD vs batoid_rubin per-CCD: the optics-model comparison.
diff_custom_bv2 = stat_custom - stat_bv2
vabs_cbv2 = np.nanpercentile(np.abs(diff_custom_bv2), 98)
fig, _ = plot_fp_map(
    diff_custom_bv2, xe, ye,
    title='custom (ts_wep per-CCD)  −  Batoid v2 (batoid per-CCD)',
    cmap='RdBu_r', vmin=-vabs_cbv2, vmax=vabs_cbv2, label='ΔZ4 [μm]',
)
_collect(fig)

# Pipeline vs the two per-CCD intrinsics.
diff_pipe_custom = stat_pipe - stat_custom
vabs_pc = np.nanpercentile(np.abs(diff_pipe_custom), 98)
fig, _ = plot_fp_map(
    diff_pipe_custom, xe, ye,
    title='pipeline  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pc, vmax=vabs_pc, label='ΔZ4 [μm]',
)
_collect(fig)

diff_pipe_bv2 = stat_pipe - stat_bv2
vabs_pbv2 = np.nanpercentile(np.abs(diff_pipe_bv2), 98)
fig, _ = plot_fp_map(
    diff_pipe_bv2, xe, ye,
    title='pipeline  −  Batoid v2 Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pbv2, vmax=vabs_pbv2, label='ΔZ4 [μm]',
)
_collect(fig)

# Transposed pipeline (fpx↔fpy per-CCD swap) vs the two per-CCD intrinsics:
# if the suspected pipeline axis swap is the only problem, these should be
# flatter than the non-transposed versions above.
diff_pipeT_custom = stat_pipeT - stat_custom
vabs_pTc = np.nanpercentile(np.abs(diff_pipeT_custom), 98)
fig, _ = plot_fp_map(
    diff_pipeT_custom, xe, ye,
    title='transposed pipeline  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pTc, vmax=vabs_pTc, label='ΔZ4 [μm]',
)
_collect(fig)

diff_pipeT_bv2 = stat_pipeT - stat_bv2
vabs_pTbv2 = np.nanpercentile(np.abs(diff_pipeT_bv2), 98)
fig, _ = plot_fp_map(
    diff_pipeT_bv2, xe, ye,
    title='transposed pipeline  −  Batoid v2 Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pTbv2, vmax=vabs_pTbv2, label='ΔZ4 [μm]',
)
_collect(fig)

In [ ]:
# Per-donut hexbin comparisons between intrinsic versions.
def _hexbin_pair(x, y, x_label, y_label, title):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    hb = ax.hexbin(x, y, gridsize=120, mincnt=1, cmap='viridis')
    lo = min(np.nanmin(x), np.nanmin(y))
    hi = max(np.nanmax(x), np.nanmax(y))
    ax.plot([lo, hi], [lo, hi], 'r-', lw=0.8, label='y = x')
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc='upper left', frameon=True)
    add_colorbar(hb).set_label('count')
    fig.tight_layout()
    return fig

pipe_sel     = Z4_pipeline_intrinsic_um[keep]
custom_sel   = Z4_custom_intrinsic_um[keep]
batoidv2_sel = Z4_batoidv2_intrinsic_um[keep]
height_sel   = height_mm[keep]

_collect(_hexbin_pair(pipe_sel, custom_sel,
    'pipeline Z4 intrinsic [μm]', 'custom Z4 intrinsic [μm]',
    'pipeline vs custom Z4 intrinsic (per donut)'))

_collect(_hexbin_pair(pipe_sel, batoidv2_sel,
    'pipeline Z4 intrinsic [μm]', 'Batoid v2 Z4 intrinsic [μm]',
    'pipeline vs Batoid v2 Z4 intrinsic (per donut)'))

_collect(_hexbin_pair(custom_sel, batoidv2_sel,
    'custom Z4 intrinsic [μm]', 'Batoid v2 Z4 intrinsic [μm]',
    'custom vs Batoid v2 Z4 intrinsic (per donut)'))

# Does pipeline − custom correlate with the height map?
fig, ax = plt.subplots(figsize=(5.5, 5.5))
hb = ax.hexbin(height_sel, (pipe_sel - custom_sel), gridsize=120, mincnt=1, cmap='viridis')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('local height [mm]')
ax.set_ylabel('pipeline − custom Z4 intrinsic [μm]')
ax.set_title('Does the pipeline intrinsic track the height map?')
add_colorbar(hb).set_label('count')
fig.tight_layout()
_collect(fig)

In [ ]:
# Residual histograms: (Z4 − linear) − I for each intrinsic I.
res_custom = Z4_linear_corr_um[keep] - Z4_custom_intrinsic_um[keep]
res_bv2    = Z4_linear_corr_um[keep] - Z4_batoidv2_intrinsic_um[keep]
res_pipe   = Z4_linear_corr_um[keep] - Z4_pipeline_intrinsic_um[keep]
res_pipeT  = Z4_linear_corr_um[keep] - Z4_pipeline_transposed_um[keep]

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
for ax, data, name in zip(
    axes,
    [res_custom, res_bv2, res_pipe, res_pipeT],
    [
        '(Z4 − linear) − custom',
        '(Z4 − linear) − Batoid v2',
        '(Z4 − linear) − pipeline',
        '(Z4 − linear) − transposed pipeline',
    ],
):
    ax.hist(data, bins=200, histtype='step')
    ax.set_title(f'{name}\nmean={np.nanmean(data):+.3f} μm,  '
                 f'std={np.nanstd(data):.3f} μm')
    ax.set_xlabel('ΔZ4 [μm]')
axes[0].set_ylabel('donuts')
fig.tight_layout()
_collect(fig)

In [ ]:
# Write every collected figure to a single PDF.
pdf_path = Path(PDF_OUTPUT)
pdf_path.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(pdf_path) as pdf:
    for f in _pdf_figs:
        pdf.savefig(f, bbox_inches='tight')
print(f'Saved {len(_pdf_figs)} pages to {pdf_path}')